## Entorno

### Dependencias

In [ ]:
!pip install -U -q PyDrive
!pip install unidecode
!pip install ultralytics
!pip install onnx
!pip install imgaug
!pip install albumentations==1.3.0
!pip install opencv-python
!pip install pybboxes
!pip install fiftyone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.5/235.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.2 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadat

In [ ]:
import pandas as pd
import re
import unidecode
import codecs
import cv2
import torch
import glob
import locale
from collections import Counter

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from google.colab import drive as dri
from oauth2client.client import GoogleCredentials
from google_drive_downloader import GoogleDriveDownloader as gdd

import sys
import inspect
import os
import zipfile
import shutil
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from ultralytics import YOLO

import seaborn as sns
import matplotlib.pyplot as plt

import fiftyone as fo
import fiftyone.zoo as foz

In [ ]:
locale.getpreferredencoding = lambda: "UTF-8"
locale.getpreferredencoding = lambda x: "UTF-8"

### Autenticacion

In [ ]:
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# dri.mount('/content/drive')

In [ ]:
def get_default_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

def to_device(data, device):
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader():
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        return
        len(self.dl)

device = get_default_device()
device

device(type='cuda')

## Fuente principal

### Main

In [ ]:
# https://docs.google.com/spreadsheets/d/1azJrrovCMYBeRasAOwh-87nVSxQPdjRs2AfBoKy97-0/edit?gid=972313100#gid=972313100

gsheetkey = "1azJrrovCMYBeRasAOwh-87nVSxQPdjRs2AfBoKy97-0"
url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=xlsx'

df = pd.read_excel(url)

display(df.head())

,Marca temporal,Dirección de correo electrónico,Nombre/s,Apellido,Comentarios sobre la entrega,Dirección del gdrive donde se encuentran las imágenes (debe ser público para lectura y no para escritura)
0,2024-06-06 19:27:16.912,lucasdemarre4@gmail.com,Lucas Federico,Demarré Estofan,NaN,https://drive.google.com/drive/folders/1oP843e...
1,2024-06-07 19:59:35.985,tomas.navarro13@live.com,Tomás,Navarro Miñón,"este drive está con un formato json, no con jp...",https://drive.google.com/drive/folders/1u9jEZc...
2,2024-06-10 10:06:35.341,facufontela98@gmail.com,Facundo,Fontela,NaN,https://drive.google.com/drive/u/0/folders/100...
3,2024-06-10 18:57:42.975,peroniantonioa@gmail.com,Antonio,Peroni,NaN,https://drive.google.com/drive/folders/1eglmMN...
4,2024-06-10 22:15:40.977,micaelapozzo25@gmail.com,Micaela,Pozzo,NaN,https://drive.google.com/drive/folders/13brsg_...


In [ ]:
folders = df['Dirección del gdrive donde se encuentran las imágenes (debe ser público para lectura y no para escritura)']

display(folders.head(2))
print(folders[0])

0    https://drive.google.com/drive/folders/1oP843e...
1    https://drive.google.com/drive/folders/1u9jEZc...
Name: Dirección del gdrive donde se encuentran las imágenes (debe ser público para lectura y no para escritura), dtype: object

https://drive.google.com/drive/folders/1oP843ex0ddgIU7voFZd-cuEddhrrFArR?usp=sharing


### Filtro

In [ ]:
def get_folder_id(folder_url):
    match = re.search(r'\/folders\/([a-zA-Z0-9_-]+)', folder_url)
    if match:
        folder_id = match.group(1)
        return folder_id
    else:
        raise ValueError(f"Cannot extract folder ID from URL: {folder_url}")

def list_files_in_folder(folder_id):
    file_list = drive.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()
    files = []
    for file in file_list:
        file_name = unidecode.unidecode(file['title'])
        file_url = f"https://drive.google.com/file/d/{file['id']}/view?usp=drive_link"
        files.append((file_name, file_url, file['id']))
    return files

In [ ]:
all_files = []

for folder_url in folders:
    folder_id = get_folder_id(folder_url)
    files = list_files_in_folder(folder_id)

    for file_name, file_url, file_id in files:

        if folder_url == 'https://drive.google.com/drive/u/0/folders/100KzDMdj7VEzSexbFfTci_wYDzSEawvv':
            if not file_name.startswith('37249_fontela_facundo'):
                continue

        all_files.append((folder_url, file_name, file_url, file_id))

file_df = pd.DataFrame(all_files, columns=["folder_url", "file_name", "file_url", "file_id"])

display(file_df)

,folder_url,file_name,file_url,file_id
0,https://drive.google.com/drive/folders/1oP843e...,43265_Lucas_Demarre_30.txt,https://drive.google.com/file/d/1d8thBSXKkaCsg...,1d8thBSXKkaCsgpJ6mmPgbgX3LRRsCk_e
1,https://drive.google.com/drive/folders/1oP843e...,43265_Lucas_Demarre_29.txt,https://drive.google.com/file/d/1z324Mnig7tqIj...,1z324Mnig7tqIjuz8ha4CWKcWxMsmk1kA
2,https://drive.google.com/drive/folders/1oP843e...,43265_Lucas_Demarre_28.txt,https://drive.google.com/file/d/13c6_6t7JXHPSg...,13c6_6t7JXHPSgY0e6QYEQJUf2xKDEIci
3,https://drive.google.com/drive/folders/1oP843e...,43265_Lucas_Demarre_27.txt,https://drive.google.com/file/d/1xo9SWuVfh4ftS...,1xo9SWuVfh4ftScqN8__Lv4N-lxbyrs0G
4,https://drive.google.com/drive/folders/1oP843e...,43265_Lucas_Demarre_26.txt,https://drive.google.com/file/d/1e0gtQeOs5vGTp...,1e0gtQeOs5vGTpM1CrD3E1ZvfH4yPPlWI
...,...,...,...,...
3310,https://drive.google.com/drive/folders/1sxvqCQ...,P51926_Leonel_Palermo_10.png,https://drive.google.com/file/d/1fOMKNK_X634ML...,1fOMKNK_X634MLEw0y0zZwqwf9OnAo9TC
3311,https://drive.google.com/drive/folders/1sxvqCQ...,P51926_Leonel_Palermo_23.png,https://drive.google.com/file/d/17hOHa7I6FpiDq...,17hOHa7I6FpiDqDvaxfZxN3SYS0uSK5mE
3312,https://drive.google.com/drive/folders/1sxvqCQ...,P51926_Leonel_Palermo_28.png,https://drive.google.com/file/d/1OlHlNC6OBfuKG...,1OlHlNC6OBfuKGZGC-2Kjzd2mzM0Mmp6-
3313,https://drive.google.com/drive/folders/1sxvqCQ...,P51926_Leonel_Palermo_26.png,https://drive.google.com/file/d/1XGJpkQh_ULiK8...,1XGJpkQh_ULiK8mOmA3c94QOU4OdpFAEm


### Control

In [ ]:
file_df.sort_values(by=["file_name"], inplace=True)
file_df.reset_index(inplace=True, drop=True)
display(file_df)

,folder_url,file_name,file_url,file_id
0,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_01.jpeg,https://drive.google.com/file/d/1A4uqu_M6gWdbP...,1A4uqu_M6gWdbPB2APkn3zz8cO0PT1C2N
1,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_01.txt,https://drive.google.com/file/d/18TX3BPnx-l5C1...,18TX3BPnx-l5C1znZG84pL65_KJyKMIkH
2,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_02.jpeg,https://drive.google.com/file/d/1oR5PQMncMNqqJ...,1oR5PQMncMNqqJf1hhOaFpo319TD9r0qv
3,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_02.txt,https://drive.google.com/file/d/14OUPT4ikCpHgx...,14OUPT4ikCpHgxN9DlwbFrPQ6j6opYncF
4,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_03.jpeg,https://drive.google.com/file/d/1qYCHZPRl31kIr...,1qYCHZPRl31kIrvRnDIuflFYM8ItQYsV5
...,...,...,...,...
3310,https://drive.google.com/drive/folders/1lOclnm...,w05576_juan_wagner_31.txt,https://drive.google.com/file/d/1FQZB0tykZTJDv...,1FQZB0tykZTJDvj-H9oIz7qfStd-TflZx
3311,https://drive.google.com/drive/folders/1lOclnm...,w05576_juan_wagner_32.png,https://drive.google.com/file/d/1KNZ4WCtcva2BJ...,1KNZ4WCtcva2BJsNurUKbC-CllBuWbUzJ
3312,https://drive.google.com/drive/folders/1lOclnm...,w05576_juan_wagner_32.txt,https://drive.google.com/file/d/1GWe__Ya4d77em...,1GWe__Ya4d77emXy3w3KevLiMthxwvi28
3313,https://drive.google.com/drive/folders/1lOclnm...,w05576_juan_wagner_33.png,https://drive.google.com/file/d/1536Qvvjei_Ay4...,1536Qvvjei_Ay4oZhw2lQpN82aDzMcmd7


In [ ]:
# Extract base file name and extension
file_df['base_file_name'] = file_df['file_name'].apply(lambda x: x.rsplit('.', 1)[0] if len(x) > 1 else '')
file_df['file_extension'] = file_df['file_name'].apply(lambda x: x.rsplit('.', 1)[1] if '.' in x else '')

file_df

,folder_url,file_name,file_url,file_id,base_file_name,file_extension
0,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_01.jpeg,https://drive.google.com/file/d/1A4uqu_M6gWdbP...,1A4uqu_M6gWdbPB2APkn3zz8cO0PT1C2N,03531_mirian_yanez_01,jpeg
1,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_01.txt,https://drive.google.com/file/d/18TX3BPnx-l5C1...,18TX3BPnx-l5C1znZG84pL65_KJyKMIkH,03531_mirian_yanez_01,txt
2,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_02.jpeg,https://drive.google.com/file/d/1oR5PQMncMNqqJ...,1oR5PQMncMNqqJf1hhOaFpo319TD9r0qv,03531_mirian_yanez_02,jpeg
3,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_02.txt,https://drive.google.com/file/d/14OUPT4ikCpHgx...,14OUPT4ikCpHgxN9DlwbFrPQ6j6opYncF,03531_mirian_yanez_02,txt
4,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_03.jpeg,https://drive.google.com/file/d/1qYCHZPRl31kIr...,1qYCHZPRl31kIrvRnDIuflFYM8ItQYsV5,03531_mirian_yanez_03,jpeg
...,...,...,...,...,...,...
3310,https://drive.google.com/drive/folders/1lOclnm...,w05576_juan_wagner_31.txt,https://drive.google.com/file/d/1FQZB0tykZTJDv...,1FQZB0tykZTJDvj-H9oIz7qfStd-TflZx,w05576_juan_wagner_31,txt
3311,https://drive.google.com/drive/folders/1lOclnm...,w05576_juan_wagner_32.png,https://drive.google.com/file/d/1KNZ4WCtcva2BJ...,1KNZ4WCtcva2BJsNurUKbC-CllBuWbUzJ,w05576_juan_wagner_32,png
3312,https://drive.google.com/drive/folders/1lOclnm...,w05576_juan_wagner_32.txt,https://drive.google.com/file/d/1GWe__Ya4d77em...,1GWe__Ya4d77emXy3w3KevLiMthxwvi28,w05576_juan_wagner_32,txt
3313,https://drive.google.com/drive/folders/1lOclnm...,w05576_juan_wagner_33.png,https://drive.google.com/file/d/1536Qvvjei_Ay4...,1536Qvvjei_Ay4oZhw2lQpN82aDzMcmd7,w05576_juan_wagner_33,png


In [ ]:
file_df.groupby('file_extension').size().reset_index(name='count')

,file_extension,count
0,jpeg,100
1,jpg,707
2,png,850
3,txt,1658


In [ ]:
allowed_extensions = ['jpeg', 'jpg', 'png', 'txt']
file_df = file_df[file_df['file_extension'].isin(allowed_extensions)]

In [ ]:
file_counts = file_df.groupby('base_file_name').size().reset_index(name='count')
file_counts[file_counts['count'] != 2].values

array([['429816_cesar_donnarumma_01', 1],
       ['429816_cesar_donnarumma_02', 1],
       ['429816_cesar_donnarumma_03', 1],
       ['429816_cesar_donnarumma_04', 1],
       ['429816_cesar_donnarumma_05', 1],
       ['429816_cesar_donnarumma_06', 1],
       ['429816_cesar_donnarumma_07', 1],
       ['429816_cesar_donnarumma_08', 1],
       ['429816_cesar_donnarumma_09', 1],
       ['429816_cesar_donnarumma_10', 1],
       ['429816_cesar_donnarumma_11', 1],
       ['429816_cesar_donnarumma_12', 1],
       ['429816_cesar_donnarumma_13', 1],
       ['429816_cesar_donnarumma_14', 1],
       ['429816_cesar_donnarumma_15', 1],
       ['429816_cesar_donnarumma_16', 1],
       ['429816_cesar_donnarumma_17', 1],
       ['429816_cesar_donnarumma_18', 1],
       ['429816_cesar_donnarumma_19', 1],
       ['429816_cesar_donnarumma_20', 1],
       ['429816_cesar_donnarumma_21', 1],
       ['429816_cesar_donnarumma_22', 1],
       ['429816_cesar_donnarumma_23', 1],
       ['429816_cesar_donnarumma_2

In [ ]:
mask = file_df['base_file_name'].str.startswith('d429816_cesar_donnarumma')
file_df.loc[mask, 'base_file_name'] = file_df.loc[mask, 'base_file_name'].str.replace('^d', '', regex=True)

In [ ]:
to_remove = file_counts[file_counts['count'] != 2]['base_file_name']
file_df = file_df[~file_df['base_file_name'].isin(to_remove)]

In [ ]:
file_df.groupby('file_extension').count()

,folder_url,file_name,file_url,file_id,base_file_name
file_extension,,,,,
jpeg,99,99,99,99,99
jpg,707,707,707,707,707
png,819,819,819,819,819
txt,1625,1625,1625,1625,1625


In [ ]:
1468 == 99+438+931

True

### Splitting

In [ ]:
file_df.head(2)

,folder_url,file_name,file_url,file_id,base_file_name,file_extension
0,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_01.jpeg,https://drive.google.com/file/d/1A4uqu_M6gWdbP...,1A4uqu_M6gWdbPB2APkn3zz8cO0PT1C2N,03531_mirian_yanez_01,jpeg
1,https://drive.google.com/drive/folders/1_LRwpr...,03531_mirian_yanez_01.txt,https://drive.google.com/file/d/18TX3BPnx-l5C1...,18TX3BPnx-l5C1znZG84pL65_KJyKMIkH,03531_mirian_yanez_01,txt


In [ ]:
file_df.shape

(3250, 6)

In [ ]:
dataset_path = '/content/dataset'
os.makedirs(dataset_path, exist_ok=True)

train_images_path = os.path.join(dataset_path, 'images/train')
val_images_path = os.path.join(dataset_path, 'images/val')
train_labels_path = os.path.join(dataset_path, 'labels/train')
val_labels_path = os.path.join(dataset_path, 'labels/val')

def clear_directory(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

clear_directory(train_images_path)
clear_directory(val_images_path)
clear_directory(train_labels_path)
clear_directory(val_labels_path)

In [ ]:
# Split the DataFrame into images and labels based on file extension
images_df = file_df[file_df['file_extension'].isin(['jpeg', 'jpg', 'png'])]
labels_df = file_df[file_df['file_extension'] == 'txt']

# Split images into training and validation sets (70% train, 30% val)
train_images_df, val_images_df = train_test_split(images_df, test_size=0.3, random_state=42)

# Get the corresponding label files for training and validation sets
train_labels_df = labels_df[labels_df['base_file_name'].isin(train_images_df['base_file_name'])]
val_labels_df = labels_df[labels_df['base_file_name'].isin(val_images_df['base_file_name'])]

# Sample 1% of the training and validation sets
# sampled_train_images_df = train_images_df.sample(frac=0.005, random_state=42)
# sampled_val_images_df = val_images_df.sample(frac=0.005, random_state=42)

sampled_train_images_df = train_images_df.sample(frac=1, random_state=42)
sampled_val_images_df = val_images_df.sample(frac=1, random_state=42)

sampled_train_labels_df = train_labels_df[train_labels_df['base_file_name'].isin(sampled_train_images_df['base_file_name'])]
sampled_val_labels_df = val_labels_df[val_labels_df['base_file_name'].isin(sampled_val_images_df['base_file_name'])]

In [ ]:
display(sampled_train_images_df.shape)
display(sampled_val_images_df.shape)
display(sampled_train_labels_df.shape)
display(sampled_val_labels_df.shape)

(1137, 6)

(488, 6)

(1137, 6)

(488, 6)

### Descarga

In [ ]:
train_images = "/content/dataset/images/train"
train_labels = "/content/dataset/labels/train"
val_images = "/content/dataset/images/val"
val_labels = "/content/dataset/labels/val"

def clear_folder(folder_path):
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            if os.path.isfile(file_path):
                os.remove(file_path)

clear_folder(train_images)
clear_folder(train_labels)
clear_folder(val_images)
clear_folder(val_labels)

In [ ]:
def download_files(df, destination_folder):
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        file_id = row['file_id']
        file_name = row['file_name']
        file_path = os.path.join(destination_folder, file_name)
        gdd.download_file_from_google_drive(file_id=file_id,
                                            dest_path=file_path,
                                            unzip=False,
                                            overwrite=False)

download_files(sampled_train_images_df, train_images_path)
download_files(sampled_val_images_df, val_images_path)
download_files(sampled_train_labels_df, train_labels_path)
download_files(sampled_val_labels_df, val_labels_path)

  0%|          | 0/1137 [00:00<?, ?it/s]

  0%|          | 1/1137 [00:02<55:25,  2.93s/it]

Done.

  0%|          | 2/1137 [00:09<1:33:16,  4.93s/it]

Done.

...

Done.

100%|█████████▉| 487/488 [19:44<00:01,  1.97s/it]

Done.

100%|██████████| 488/488 [19:47<00:00,  2.43s/it]

Done.


### Formato

In [ ]:
def convert_segment_to_bbox(segment_annotation):
    try:
        coords = list(map(float, segment_annotation.split()[1:]))  # Skip the first value (class)
        x_coords = coords[0::2]
        y_coords = coords[1::2]

        min_x = min(x_coords)
        max_x = max(x_coords)
        min_y = min(y_coords)
        max_y = max(y_coords)

        x_center = (min_x + max_x) / 2
        y_center = (min_y + max_y) / 2
        width = max_x - min_x
        height = max_y - min_y

        return f"{segment_annotation.split()[0]} {x_center} {y_center} {width} {height}"

    except ValueError as e:
        print(f"Error converting segment to bbox: {e}, in annotation: {segment_annotation}")
        return None

def process_annotation_file(file_path):
    with codecs.open(file_path, 'r', encoding='ascii', errors='ignore') as file:
        lines = file.readlines()

    new_lines = []
    for line in lines:
        if len(line.split()) > 5:  # Assuming segment annotations have more than 5 values
            new_line = convert_segment_to_bbox(line.strip())
            if new_line:
                new_lines.append(new_line)
        else:
            new_lines.append(line.strip())  # already in bbox format

    with open(file_path, 'w') as file:
        file.write("\n".join(new_lines) + "\n")

def process_annotations_folder(folder_path):
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            process_annotation_file(os.path.join(folder_path, filename))

process_annotations_folder(train_labels_path)
process_annotations_folder(val_labels_path)

Error converting segment to bbox: could not convert string to float: 'html><html><head><title>Google', in annotation: <!DOCTYPE html><html><head><title>Google Drive - Virus scan warning</title><meta http-equiv="content-type" content="text/html; charset=utf-8"/><style nonce="735CpI24rV599iHSBROGmQ">.goog-link-button{position:relative;color:#15c;text-decoration:underline;cursor:pointer}.goog-link-button-disabled{color:#ccc;text-decoration:none;cursor:default}body{color:#222;font:normal 13px/1.4 arial,sans-serif;margin:0}.grecaptcha-badge{visibility:hidden}.uc-main{padding-top:50px;text-align:center}#uc-dl-icon{display:inline-block;margin-top:16px;padding-right:1em;vertical-align:top}#uc-text{display:inline-block;max-width:68ex;text-align:left}.uc-error-caption,.uc-warning-caption{color:#222;font-size:16px}#uc-download-link{text-decoration:none}.uc-name-size a{color:#15c;text-decoration:none}.uc-name-size a:visited{color:#61c;text-decoration:none}.uc-name-size a:active{color:#d14836;text-

### Extensiones

In [ ]:
def normalize_images(input_dir):
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):

            file_path = os.path.join(input_dir, filename)

            # Check if the image file exists and is readable
            if not os.path.exists(file_path):
                print(f"Error: Image file not found: {file_path}")
                continue

            image = cv2.imread(file_path)

            # Check if the image was loaded successfully
            if image is None:
                print(f"Error: Could not read image file: {file_path}")
                continue

            base_filename, _ = os.path.splitext(filename)
            output_filename = base_filename + '.png'
            output_path = os.path.join(input_dir, output_filename)

            cv2.imwrite(output_path, image)

            if not filename.lower().endswith('.png'):
                os.remove(file_path)

normalize_images(train_images_path)
normalize_images(val_images_path)

Error: Could not read image file: /content/dataset/images/train/R-45829_Abril_Rodriguez_04.png
Error: Could not read image file: /content/dataset/images/train/64191_Sofia_BrizuelaCipolletti_10.jpg
Error: Could not read image file: /content/dataset/images/train/s56189_salvador_sanchez_45.jpg
Error: Could not read image file: /content/dataset/images/train/f37478_santiago_ferrero_06.jpg
Error: Could not read image file: /content/dataset/images/train/s56189_salvador_sanchez_153.jpg
Error: Could not read image file: /content/dataset/images/train/71846_Brisa_Menescaldi_05.png
Error: Could not read image file: /content/dataset/images/train/51934_pablo_pistelli_27.png
Error: Could not read image file: /content/dataset/images/train/51659_antonio_peroni_19.jpg
Error: Could not read image file: /content/dataset/images/train/P51951_pablo_pistarelli_01.png
Error: Could not read image file: /content/dataset/images/train/71846_Brisa_Menescaldi_03.png
Error: Could not read image file: /content/dataset

## Descarga

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
folder_path = '/content/dataset'
zip_path = '/content/drive/MyDrive/carta_base.zip'

# Create a zip file from the folder
shutil.make_archive(base_name=zip_path.replace('.zip', ''), format='zip', root_dir=folder_path)

print(f"Folder has been zipped and saved to {zip_path}")

# List files in the destination directory to verify
destination_directory = '/content/drive/MyDrive'
print("Files in destination directory:")
for file_name in os.listdir(destination_directory):
    print(file_name)

Folder has been zipped and saved to /content/drive/MyDrive/carta_base.zip
Files in destination directory:
carta_base.zip
